# 🥈 Silver — `silver.flights_clean` + `silver.dim_airports`

Dados confiáveis: limpos, padronizados, tipados e validados.

|  |  |
|---|---|
| **Lê** | `bronze.flights_raw` |
| **Grava** | `silver.flights_clean` (célula 1) · `silver.dim_airports` (célula 2) |
| **Regido por** | `silver_rules.md` — cada regra tem lá problema, evidência, decisão e validação |

### Regras aplicadas na célula 1

| Regra | O que faz |
|---|---|
| 03 | Flag `is_extreme_delay` (`arr_delay > 180`) — sinaliza o outlier, não exclui |
| 06 | `op_carrier_fl_num` → `integer` (é identificador, não medida) |
| 07 | `TRIM` preventivo no texto + maiúsculas nos códigos |
| 09 | `dropDuplicates()` — proteção; no sample e no completo removeu 0 |
| 10 | Validação estrutural: `fl_date`/`origin`/`dest` não nulos, `origin != dest`, `distance > 0` |

As regras **01, 02, 04, 05 e 08 são decisões de _não_ transformar** (manter nulos legítimos, manter cancelado com `dep_time` preenchido, manter horário `hhmm` como inteiro), por isso não aparecem como código.

### Célula 2 — dimensão de aeroportos (Regra 11)

Existe para o dashboard e o agente de IA mostrarem "Atlanta, GA" e não só "ATL". **A sigla continua sendo a chave:** o nome não é único — "Chicago, IL" é ORD *e* MDW, "Houston, TX" é IAH *e* HOU. A célula imprime a prova disso antes de terminar.

> **Execução com o dataset completo (Etapa 15):** 7.079.081 → 7.079.081 — 0 inválidos, 0 duplicatas, 21,5s.


In [ ]:
# ============================================================
# ETAPA 15 — SILVER (DATASET COMPLETO)
# Notebook independente — lê direto da tabela bronze.flights_raw
# ============================================================

import time
from pyspark.sql import functions as F

inicio_execucao = time.time()

# Ler a Bronze (já deve estar criada por um notebook anterior)
df = spark.table("bronze.flights_raw")
contagem_bronze = df.count()
print("Registros na Bronze:", contagem_bronze)

# Regra 06 — op_carrier_fl_num: converter para integer
df = df.withColumn("op_carrier_fl_num", F.col("op_carrier_fl_num").cast("int"))

# Regra 07 — Padronização de texto (trim preventivo)
colunas_texto = [
    "op_unique_carrier", "origin", "dest",
    "origin_city_name", "origin_state_nm",
    "dest_city_name", "dest_state_nm",
    "cancellation_code"
]
for coluna in colunas_texto:
    df = df.withColumn(coluna, F.trim(F.col(coluna)))

for coluna in ["op_unique_carrier", "origin", "dest", "cancellation_code"]:
    df = df.withColumn(coluna, F.upper(F.col(coluna)))

# Regra 03 — Flag de atraso extremo
df = df.withColumn(
    "is_extreme_delay",
    F.when(F.col("arr_delay") > 180, True).otherwise(False)
)

# Regra 10 — Validações estruturais obrigatórias
condicao_valida = (
    F.col("fl_date").isNotNull() &
    F.col("origin").isNotNull() &
    F.col("dest").isNotNull() &
    (F.col("origin") != F.col("dest")) &
    (F.col("distance") > 0)
)

df_invalidos = df.filter(~condicao_valida)
qtd_invalidos = df_invalidos.count()
df_validos = df.filter(condicao_valida)

print("Registros que falharam na validação estrutural:", qtd_invalidos)

# Regra 09 — Remover duplicatas exatas
contagem_antes_dedup = df_validos.count()
df_silver = df_validos.dropDuplicates()
contagem_depois_dedup = df_silver.count()

print("Registros antes de remover duplicatas:", contagem_antes_dedup)
print("Registros depois de remover duplicatas:", contagem_depois_dedup)
print("Duplicatas removidas:", contagem_antes_dedup - contagem_depois_dedup)

# Salvar como Delta
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.flights_clean")

contagem_silver = df_silver.count()

print("\n=== RESUMO SILVER (DATASET COMPLETO) ===")
print("Registros na Bronze:", contagem_bronze)
print("Registros inválidos removidos:", qtd_invalidos)
print("Duplicatas removidas:", contagem_antes_dedup - contagem_depois_dedup)
print("Registros finais na Silver:", contagem_silver)
print(f"Tempo de execução: {round(time.time() - inicio_execucao, 1)}s")


In [ ]:
# ============================================================
# ETAPA 15 — SILVER · CÉLULA 2
# silver.dim_airports — dimensão de aeroportos (código + nome)
# ------------------------------------------------------------
# POR QUE UMA DIMENSÃO, E NÃO TROCAR O CÓDIGO PELO NOME:
# o código IATA é a chave (único, estável, 3 letras). O nome da cidade
# NÃO é único — "Chicago, IL" é ORD e MDW, "Houston, TX" é IAH e HOU,
# "Washington, DC" é DCA e IAD. Se a Gold agrupasse por nome, esses
# aeroportos seriam somados no mesmo registro e o número ficaria errado.
# Então: a chave continua sendo o código; o nome entra como atributo.
#
# O nome vem das colunas origin_city_name / dest_city_name do próprio
# dataset BTS (o dataset não traz o nome oficial do terminal). Nenhum
# dado externo é digitado à mão — tudo é rastreável até a Bronze.
# ============================================================

from pyspark.sql import functions as F, Window

df_silver = spark.table("silver.flights_clean")

# 1) Um aeroporto aparece ora como origem, ora como destino.
#    Empilhamos os dois lados para não perder nenhum código.
origens = df_silver.select(
    F.col("origin").alias("airport_code"),
    F.col("origin_city_name").alias("airport_name"),
    F.col("origin_state_nm").alias("airport_state"),
)

destinos = df_silver.select(
    F.col("dest").alias("airport_code"),
    F.col("dest_city_name").alias("airport_name"),
    F.col("dest_state_nm").alias("airport_state"),
)

todos = origens.unionByName(destinos).filter(
    F.col("airport_code").isNotNull() & F.col("airport_name").isNotNull()
)

# 2) O mesmo código pode aparecer com grafias diferentes da cidade em
#    alguns registros. Contamos as ocorrências e ficamos com a grafia
#    mais frequente — assim a dimensão tem exatamente 1 linha por código.
contagem = todos.groupBy("airport_code", "airport_name", "airport_state").agg(
    F.count("*").alias("ocorrencias")
)

janela = Window.partitionBy("airport_code").orderBy(
    F.col("ocorrencias").desc(), F.col("airport_name").asc()
)

df_dim = (
    contagem
    .withColumn("posicao", F.row_number().over(janela))
    .filter(F.col("posicao") == 1)
    .drop("posicao", "ocorrencias")
    # "Atlanta, GA" -> airport_city = "Atlanta" (útil para filtro por cidade)
    .withColumn("airport_city", F.trim(F.split(F.col("airport_name"), ",").getItem(0)))
    # rótulo pronto para o dashboard e para o agente de IA: "ATL - Atlanta, GA"
    .withColumn(
        "airport_label",
        F.concat_ws(" - ", F.col("airport_code"), F.col("airport_name")),
    )
    .select("airport_code", "airport_name", "airport_city", "airport_state", "airport_label")
    .orderBy("airport_code")
)

df_dim.write.format("delta").mode("overwrite").saveAsTable("silver.dim_airports")

# ------------------------------------------------------------
# QA da dimensão
# ------------------------------------------------------------
qtd_dim = df_dim.count()
qtd_codigos = todos.select("airport_code").distinct().count()

print("Aeroportos distintos nos voos:", qtd_codigos)
print("Linhas em silver.dim_airports:", qtd_dim)
print("Duplicidade de chave:", "OK (1 linha por código)" if qtd_dim == qtd_codigos else "ATENÇÃO")

# Prova de que o código não podia ser substituído pelo nome:
print("\nCidades atendidas por mais de um aeroporto (nome NÃO é chave):")
(
    df_dim.groupBy("airport_name")
    .agg(F.count("*").alias("qtd_aeroportos"), F.collect_list("airport_code").alias("codigos"))
    .filter(F.col("qtd_aeroportos") > 1)
    .orderBy(F.col("qtd_aeroportos").desc())
    .show(10, truncate=False)
)

display(df_dim.limit(10))